In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv(r'C:\Users\gurus\OneDrive\Desktop\study\project\3rd project\youtube_ad_revenue_dataset.csv')
data.shape
data.head(5)
df = data.copy()
df.head(5)

In [ ]:
# missing value 
df.isnull().sum()

In [ ]:
# data type 
df.dtypes

In [ ]:
#description
df.describe()

In [ ]:
df['likes'] = df['likes'].fillna(df.groupby('category')['likes'].transform('mean'))
df['comments'] = df['comments'].fillna(df.groupby(['category','country','device'])['comments'].transform('mean'))
df['watch_time_minutes'] = df['watch_time_minutes'].fillna(df.groupby(['category','country','device'])['watch_time_minutes'].transform('mean'))

In [ ]:
df['likes'].dtype
df['likes'].isnull().sum()

In [ ]:
df.isnull().sum()

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['date'].dtype
df.date.head(5)

In [ ]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['quarter'] = df['date'].dt.quarter

In [ ]:
df.drop(['video_id','date'],axis = 1,inplace = True)

In [ ]:
df.head(2)

In [ ]:
df.dtypes

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.histplot(df['ad_revenue_usd'],bins=50,color='blue')
plt.title('Ad revenue Discription ')
plt.xlabel('revenue')
plt.ylabel("count")
plt.show()

In [ ]:
sns.boxplot(x=df['ad_revenue_usd'],color='orange')
plt.title('Ad revenue Discription ')
plt.xlabel('revenue')
plt.show()

In [ ]:
sns.heatmap(df.corr(numeric_only=True),annot=True,fmt='.2f',cmap='coolwarm')
plt.title('correlation headmap')
plt.show()

In [ ]:
df.drop(columns=['quarter'], inplace=True)

In [ ]:
df['watch_revenue_interaction'] = df['watch_time_minutes'] * df['ad_revenue_usd']

In [ ]:
df['like_rate'] = df['likes'] / (df['views'] + 1)

In [ ]:
import numpy as np
for col in ['views', 'likes', 'comments', 'ad_revenue_usd']:
    df[f'log_{col}'] = np.log1p(df[col])

In [ ]:
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

In [ ]:
cat_rev = df.groupby('category')['ad_revenue_usd'].mean().sort_values(ascending=False)
sns.barplot(x=cat_rev.index,y=cat_rev.values,palette='viridis')
plt.title("cat_wies")
plt.xlabel('cat')
plt.ylabel('avg_ren')
plt.xticks(rotation =45)

In [ ]:
cat_rev = df.groupby('country')['ad_revenue_usd'].mean().sort_values(ascending=False)
sns.barplot(x=cat_rev.index,y=cat_rev.values,palette='viridis')
plt.title("cat_wies")
plt.xlabel('cat')
plt.ylabel('avg_ren')
plt.xticks(rotation =45)
plt.show()

In [ ]:
cat_rev = df.groupby('device')['ad_revenue_usd'].mean().sort_values(ascending=False)
sns.barplot(x=cat_rev.index,y=cat_rev.values,palette='viridis')
plt.title("cat_wies")
plt.xlabel('cat')
plt.ylabel('avg_ren')
plt.xticks(rotation =45)
plt.show()

In [ ]:
df.head(2)

In [ ]:
print(df.columns.tolist())

In [ ]:
 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

In [ ]:
# Step 1: Feature Engineering
df['log_ad_revenue_usd'] = np.log1p(df['ad_revenue_usd'])
df['like_rate'] = df['likes'] / (df['views'] + 1)
df['comment_rate'] = df['comments'] / (df['views'] + 1)

for col in ['views', 'likes', 'comments']:
    df[f'log_{col}'] = np.log1p(df[col])

In [ ]:
# Step 2: 5 Features மட்டும்
X = df[['watch_time_minutes',
        'log_likes',
        'like_rate',
        'video_length_minutes',
        'log_views']]
print(X.head(2))

In [ ]:
y = df['log_ad_revenue_usd']

# Step 3: Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# ✅ Step 4: Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# ✅ Step 5: Train
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

In [ ]:
# ✅ Step 6: Evaluate
y_pred = model.predict(X_test_scaled)
y_pred_actual = np.expm1(y_pred)
y_test_actual = np.expm1(y_test)

In [ ]:
print(f'MAE  : {mean_absolute_error(y_test_actual, y_pred_actual):.2f}')
print(f'R²   : {r2_score(y_test_actual, y_pred_actual):.4f}')


In [ ]:



# Step 6: Save
joblib.dump(model, 'final_ads_revenue_model.pkl')
print("Model Saved! ✅")

In [ ]:
residuals = y_test_actual - y_pred_actual

import matplotlib.pyplot as plt

plt.hist(residuals, bins=30)
plt.xlabel("Prediction Error")
plt.ylabel("Frequency")
plt.title("Residual Distribution")
plt.show()

In [ ]:
plt.scatter(y_test_actual, y_pred_actual)
plt.xlabel("Actual Revenue")
plt.ylabel("Predicted Revenue")
plt.title("Actual vs Predicted Revenue")
plt.show()

In [ ]:
print(f'MAE  : {mean_absolute_error(y_test_actual, y_pred_actual):.2f}')
print(f'R²   : {r2_score(y_test_actual, y_pred_actual):.4f}')


In [ ]:
print(residuals.describe())

In [ ]:
print(df['ad_revenue_usd'].describe())

In [ ]:
import matplotlib.pyplot as plt

# Feature Importance எடுக்கோம்
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance)

# Graph போடோம்
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], 
         feature_importance['importance'])
plt.title('Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
!pip install xgboost

In [ ]:
import sys
!{sys.executable} -m pip install xgboost

In [ ]:
import xgboost
print(xgboost.__version__)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb
from xgboost import XGBRegressor
import joblib

# ✅ Step 1: Feature Engineering
df['log_ad_revenue_usd'] = np.log1p(df['ad_revenue_usd'])
df['like_rate'] = df['likes'] / (df['views'] + 1)

for col in ['views', 'likes']:
    df[f'log_{col}'] = np.log1p(df[col])

# ✅ Step 2: Features & Target
X = df[['watch_time_minutes',
        'log_likes',
        'like_rate',
        'video_length_minutes',
        'log_views']]

y = df['log_ad_revenue_usd']

# ✅ Step 3: Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ✅ Step 4: Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ✅ Step 5: XGBoost Train
model_xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)
model_xgb.fit(X_train_scaled, y_train)

# ✅ Step 6: Evaluate
y_pred = model_xgb.predict(X_test_scaled)
y_pred_actual = np.expm1(y_pred)
y_test_actual = np.expm1(y_test)

print(f'MAE  : {mean_absolute_error(y_test_actual, y_pred_actual):.2f}')
print(f'R²   : {r2_score(y_test_actual, y_pred_actual):.4f}')

# ✅ Step 7: Save
joblib.dump(model_xgb, 'final_ads_revenue_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print("XGBoost Model Saved! ✅")

In [ ]:
import os
size = os.path.getsize('final_ads_revenue_model.pkl')
print(f"Model size: {size / 1024 / 1024:.2f} MB")

In [ ]:
import os
print(f"Model: {os.path.getsize('final_ads_revenue_model.pkl') / 1024 / 1024:.2f} MB")
print(f"Scaler: {os.path.getsize('scaler.pkl') / 1024 / 1024:.2f} MB")